<a href="https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Paper Finding #1: "Brand-stature forms a three-tier visibility ladder, where Tier 1 global household brands appear in 72.9% of unbranded AI search answers on day-1, Tier 2 in 43.6%, and Tier 3 in 11.4%."

Where does the label/tier come from? Brands are classified into tiers using a hand-coded rubric based on external web prominence signals (Wikipedia article length $\ge 5,000$ words, top-20 mainstream press mentions, and $\ge \text{Series C}$ funding or public company status) more_horiz.

Does the validation design support the claim? Methodology Question: Because the rubric uses web prominence proxies to define the tiers, the day-1 visibility ladder is an observational quantification of existing web authority rather than an independent causal discovery. Does the observational snapshot control for survivorship bias across older brand domains? As the paper transparently acknowledges in its limitations section, without a randomized trial or a multi-rater inter-rater audit (e.g., Cohen's $\kappa$), brand-stature tiering remains a descriptive baseline rather than proof that shifting a brand's tier causes an immediate jump in AI visibility.


Paper Finding #2: "Freshness is the strongest growth signal: updating mature 365+ day content yields a 1.6x health boost and a 52x impression boost compared to untouched stale pages".

Where does the label come from? The freshness label is derived from the time elapsed since the last recorded workflow edit (days_since_last_update).

Does the validation design support the claim? Methodology Question: The 52x impression lift is measured across a non-randomized observational cohort. Does the selection design control for editorial selection bias—the fact that content teams selectively choose to refresh historically strong, high-demand assets while abandoning weak ones
? Without an out-of-sample randomized controlled trial (RCT) comparing updated vs. non-updated pages across matched baseline traffic strata, the 52x lift represents an upper-bound association rather than a guaranteed causal multiplier for every stale page.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Validation Split Integrity Audit (Random Row Split vs. Client-Grouped Split):
Taking a look at Week-5 model, we evaluate the Random Forest classifier under two distinct split strategies on the exact same dataset
:

BEFORE (Random Row Split): Yields an artificially high Precision@50 of ~74.0% and ROC AUC of 0.750. Because multiple pages from the same client domain appear in both training and test sets, the model memorizes client-specific domain authority and baseline traffic levels.

AFTER (Client-Grouped Split): Holding out entire client domains (GroupShuffleSplit on client_id) yields an honest Precision@50 of 54.0% and ROC AUC of 0.611.
This uncorrupted benchmark measures true decision-support performance when deploying models to newly onboarded websites


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split


# Helper function for Precision@K metric
def precision_at_k(y_true, y_probs, k=50):
  eval_df = pd.DataFrame({'y_true': y_true, 'y_prob': y_probs})
  top_k = eval_df.sort_values(by='y_prob', ascending=False).head(k)
  return float(top_k['y_true'].mean())


# 1. Load starter dataset and apply availability filter
df_raw = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df = df_raw[
    (df_raw['impressions_90d'] > 0) & (df_raw['content_age_days'] >= 90)
].copy()
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['log_impressions'] = np.log1p(df['impressions_90d'])
df['log_sessions'] = np.log1p(df['sessions_90d'])

feature_cols = [
    'impressions_90d',
    'sessions_90d',
    'avg_position',
    'ctr',
    'content_age_days',
    'word_count',
    'log_impressions',
    'log_sessions',
]
X = df[feature_cols].fillna(0)
y = df['is_declining']
groups = df['client_id']

# --- BEFORE: Random Row Split (Leaky / Shared Clients) ---
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y, test_size=0.20, random_state=42
)
rf_naive = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42
)
rf_naive.fit(X_tr_r, y_tr_r)
p_naive = rf_naive.predict_proba(X_te_r)[:, 1]

# --- AFTER: Client-Grouped Split (Honest / Held-Out Clients) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
X_tr_g, X_te_g = X.iloc[tr_idx], X.iloc[te_idx]
y_tr_g, y_te_g = y.iloc[tr_idx], y.iloc[te_idx]

rf_honest = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42
)
rf_honest.fit(X_tr_g, y_tr_g)
p_honest = rf_honest.predict_proba(X_te_g)[:, 1]

# Display Split Comparison Table
split_comp = pd.DataFrame([
    {
        'Split Type': 'BEFORE (Random Row Split)',
        'ROC AUC': round(roc_auc_score(y_te_r, p_naive), 3),
        'Avg Precision': round(average_precision_score(y_te_r, p_naive), 3),
        'Precision@50': round(precision_at_k(y_te_r, p_naive, 50), 3),
        'Validation Discipline': 'Leaky (Shared Clients)',
    },
    {
        'Split Type': 'AFTER (Client-Grouped Split)',
        'ROC AUC': round(roc_auc_score(y_te_g, p_honest), 3),
        'Avg Precision': round(average_precision_score(y_te_g, p_honest), 3),
        'Precision@50': round(precision_at_k(y_te_g, p_honest, 50), 3),
        'Validation Discipline': 'Honest (Unseen Clients)',
    },
])

print('SECTION 2: VALIDATION SPLIT BEFORE / AFTER COMPARISON')
print(split_comp.to_string(index=False))

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Feature Importance Ranking:

avg_position (21.54%) & impressions_90d / log_impressions (34.92% combined): Search position volatility and search volume exposure account for over 56% of model decisions.

content_age_days (16.24%) & word_count (12.48%): Metadata signals provide secondary context.

ctr (6.31%) & sessions_90d / log_sessions (8.52% combined): Click and session traffic refine probabilities.

Leakage Verification:
Confirmed zero inclusion of circular product decision flags (health_score, priority_score, refresh_tier)
 or post-decision evaluation window metrics.


Failure Mode Analysis (Real Error Examples):

1. False Positives (Over-Predicted Decay — High Probability, Actual Stable Target y_true == 0):

Example: A page with 2,639 impressions, position 7.2, and age 106 days received a 74.5% decay probability despite remaining stable (y_true == 0).
Root Cause: The model over-indexed on position fluctuations on high-demand pages, mistaking external SERP layout changes (sponsored ad expansions or Google AI Overviews) for internal content decay.

2. False Negatives (Missed Decay — Low Probability, Declining Target y_true == 1):
Example: Low-volume pages (2 and 4 impressions) received low decay probabilities (14.8% and 30.1%) despite actively declining (y_true ==

Root Cause: The model deprioritizes low-traffic niche articles because absolute impression drops are small and masked by sampling noise

In [3]:
import pandas as pd

# 1. Feature importances from Week-5 execution
feat_imp_df = pd.DataFrame({
    'Feature': [
        'avg_position',
        'impressions_90d',
        'content_age_days',
        'log_impressions',
        'word_count',
        'ctr',
        'sessions_90d',
        'log_sessions',
    ],
    'Importance_Pct': [21.54, 20.17, 16.24, 14.75, 12.48, 6.31, 4.26, 4.26],
})

# 2. False Positives (High model probability, actual y_true == 0)
fp_df = pd.DataFrame({
    'impressions_90d': [307, 2426, 2639],
    'avg_position': [39.8, 30.0, 7.2],
    'content_age_days': [238, 300, 106],
    'rf_prob': [0.800367, 0.751236, 0.745028],
    'y_true': [0, 0, 0],
})

# 3. False Negatives (Low model probability, actual y_true == 1)
fn_df = pd.DataFrame({
    'impressions_90d': [4, 2, 1035],
    'avg_position': [36.3, 7.5, 23.6],
    'content_age_days': [348, 126, 545],
    'rf_prob': [0.300990, 0.148778, 0.322274],
    'y_true': [1, 1, 1],
})

# Print Section 3 Tables
print("=== FEATURE IMPORTANCE BREAKDOWN ===")
print(feat_imp_df.to_string(index=False))

print("\n=== ERROR ANALYSIS: FALSE POSITIVES (Model Over-predicted Decay) ===")
print(fp_df.to_string(index=False))

print("\n=== ERROR ANALYSIS: FALSE NEGATIVES (Model Missed Real Decay) ===")
print(fn_df.to_string(index=False))

=== FEATURE IMPORTANCE BREAKDOWN ===
         Feature  Importance_Pct
    avg_position           21.54
 impressions_90d           20.17
content_age_days           16.24
 log_impressions           14.75
      word_count           12.48
             ctr            6.31
    sessions_90d            4.26
    log_sessions            4.26

=== ERROR ANALYSIS: FALSE POSITIVES (Model Over-predicted Decay) ===
 impressions_90d  avg_position  content_age_days  rf_prob  y_true
             307          39.8               238 0.800367       0
            2426          30.0               300 0.751236       0
            2639           7.2               106 0.745028       0

=== ERROR ANALYSIS: FALSE NEGATIVES (Model Missed Real Decay) ===
 impressions_90d  avg_position  content_age_days  rf_prob  y_true
               4          36.3               348 0.300990       1
               2           7.5               126 0.148778       1
            1035          23.6               545 0.322274       1


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The Random Forest model proves that updating stale content causes search traffic recovery. We observed that Random Forest scoring increased Precision@50 from 38.0% to 54.0% on a 20% client-grouped holdout test set. Proving traffic recovery requires experimental A/B testing or causal designs; observational data supports directional prioritization only.
The algorithm accurately predicts Google search ranking drops with zero error. The model serves as a decision-support ranking tool, prioritizing high-volume pages experiencing position volatility for human editorial review. False positives (~46%) occur due to external SERP layout shifts, requiring human editorial oversight.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.